# Axion Dark Matter PINN — Single Field, Universal Mass Range

## เป้าหมาย

ไฟล์นี้เน้น **axion 1 ตัว** แต่รองรับ **ทุกมวล ทุกช่วงเวลา** โดยอัตโนมัติ

---

### กลยุทธ์รองรับทุกมวล

| มวล (code units) | ระบอบ | กลยุทธ์ |
|---|---|---|
| $m \ll H_0$ (~1–10) | Slow-roll (ยังไม่สั่น) | Train ตรงๆ ที่ domain $t \in [0,1]$ |
| $m \sim H_0$ (~10–500) | Onset of oscillation | Train ตรงๆ + adaptive collocation ใกล้ $a_{osc}$ |
| $m \gg H_0$ (~500–1e9+) | Rapid oscillations | **Rescaling**: simulate ที่ $m_{sim}$ แล้ว map กลับ |

**Key idea (Rescaling):** Klein-Gordon + Friedmann ระบบมี self-similarity ภายใต้
$$m \to \lambda m, \quad t \to t/\lambda$$
ดังนั้น train ที่ $m_{sim}$ แล้ว stretch แกน $t$ กลับด้วย $\lambda = \sqrt{m_{target}/m_{sim}}$

---

### Physics

$$H = \sqrt{\frac{\rho_{tot}}{3}}, \quad \rho_{tot} = \frac{1}{2}\dot\phi^2 + \frac{1}{2}m^2\phi^2 + \frac{\rho_{m,0}}{a^3} + \frac{\rho_{r,0}}{a^4} + \rho_\Lambda$$

$$\ddot\phi + 3H\dot\phi + m^2\phi = 0, \qquad \dot a = aH$$

## 1. Imports & Setup

In [ ]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# ---------- Reproducibility ----------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float64   # float64 สำคัญมากสำหรับ stiff ODEs

print(f"Device  : {DEVICE}")
print(f"Dtype   : {DTYPE}")
print(f"PyTorch : {torch.__version__}")
print(f"NumPy   : {np.__version__}")

## 2. Cosmological Parameters & Mass Regime Detector

`MassRegimeConfig` วิเคราะห์มวลที่ผู้ใช้ระบุ แล้วเลือก:
- **Direct mode** → สำหรับ $m_{code} \leq 500$ (N_osc ไม่มากเกินไป)
- **Rescaling mode** → สำหรับ $m_{code} > 500$ (สั่นเร็วเกิน → simulate ที่ $m_{sim}$ แล้ว rescale)

In [ ]:
# =====================================================
# Cosmological parameters (reduced Planck units: 8πG/3 = 1)
# H0 = 1 in code units
# =====================================================
RHO_M0 = 0.81         # Ω_m * 3H0²
RHO_R0 = 0.00027138   # Ω_r * 3H0²
RHO_L0 = 2.19         # Ω_Λ * 3H0²

# Standard cosmological checks
z_eq = RHO_M0 / RHO_R0 - 1
a_eq = 1.0 / (1.0 + z_eq)
print(f"Matter–Radiation equality:  z_eq = {z_eq:.0f}  |  a_eq = {a_eq:.4e}")

# Physical unit conversion
H0_eV = 6.582e-16 * 2.184e-18   # ℏ * H0  [eV]
print(f"H0 in eV: ℏ·H0 = {H0_eV:.4e} eV  (m_code=1 → m_a = {H0_eV:.3e} eV)\n")

# =====================================================
# Mass Regime Analyzer
# =====================================================
MAX_N_OSC_DIRECT = 500   # max tolerable N_osc for direct training
M_SIM_DEFAULT    = 100.0 # reference simulation mass for rescaling mode


def analyze_mass_regime(m_target: float, t_end: float = 1.0,
                        max_n_osc: float = MAX_N_OSC_DIRECT,
                        m_sim_default: float = M_SIM_DEFAULT) -> dict:
    """
    Given target mass m_target (code units) and integration domain [0, t_end],
    decide whether to use direct training or rescaling strategy.

    Returns a config dict with all derived simulation parameters.
    """
    n_osc = m_target / (2 * np.pi) * t_end  # approximate number of oscillations
    a_osc = (3 * RHO_R0 / m_target**2) ** 0.25

    if n_osc <= max_n_osc:
        # ── Direct mode ────────────────────────────────────────────
        mode = 'direct'
        lam  = 1.0
        m_sim = m_target
        t_end_sim = t_end
        rho_scale = 1.0
    else:
        # ── Rescaling mode ─────────────────────────────────────────
        # Choose m_sim so that N_osc ≈ max_n_osc / 2 (comfortable headroom)
        target_n_osc_sim = max_n_osc / 2.0
        # N_osc_sim = m_sim/(2π) * t_end_sim = m_sim/(2π) * (t_end/λ) = m_sim/(2π) * t_end / sqrt(m_target/m_sim)
        # Solve: m_sim / sqrt(m_target/m_sim) = 2π * target_n_osc_sim / t_end
        # => m_sim^(3/2) / sqrt(m_target) = constant => m_sim = (const * sqrt(m_target))^(2/3)
        C = 2 * np.pi * target_n_osc_sim / t_end
        m_sim = (C * np.sqrt(m_target)) ** (2.0 / 3.0)
        m_sim = max(m_sim, m_sim_default)   # floor at m_sim_default

        mode  = 'rescaling'
        lam   = np.sqrt(m_target / m_sim)
        t_end_sim  = t_end / lam
        rho_scale  = lam ** 2   # ρ_phys = ρ_sim * λ²

    regime = ('slow-roll'   if m_target < 3.0 else
              'onset'       if m_target < 50.0 else
              'oscillating' if m_target < 500.0 else
              'rapid-osc')

    return dict(
        mode       = mode,
        regime     = regime,
        m_target   = m_target,
        m_sim      = m_sim,
        lam        = lam,
        t_end      = t_end,
        t_end_sim  = t_end_sim,
        a_osc      = a_osc,
        n_osc      = n_osc,
        rho_scale  = rho_scale,
    )


def print_mass_summary(cfg: dict):
    print("=" * 65)
    print(f"  Mass Analysis:  m_target = {cfg['m_target']:.3g}  |  regime = {cfg['regime']}")
    print(f"  Mode           : {cfg['mode']}")
    print(f"  N_osc (physical): {cfg['n_osc']:.2e}  over t∈[0,{cfg['t_end']}]")
    print(f"  a_osc          : {cfg['a_osc']:.3e}")
    if cfg['mode'] == 'rescaling':
        print(f"  m_sim          : {cfg['m_sim']:.3g}  (λ = {cfg['lam']:.3g})")
        print(f"  t_end_sim      : {cfg['t_end_sim']:.3e}  (simulation domain)")
        print(f"  ρ rescale      : × λ² = {cfg['rho_scale']:.3e}")
    m_eV = cfg['m_target'] * H0_eV
    print(f"  Physical mass  : {m_eV:.3e} eV")
    print("=" * 65)


# ── Quick overview of regimes ──────────────────────────────────
print(f"{'m_code':>10}  {'m_a (eV)':>12}  {'N_osc':>10}  {'mode':>12}  {'regime':>14}")
print("-" * 65)
for mc in [1, 10, 70, 200, 500, 1e4, 1e6, 1e8, 1e10]:
    rc = analyze_mass_regime(mc)
    print(f"{mc:>10.1e}  {mc*H0_eV:>12.3e}  {rc['n_osc']:>10.2e}  {rc['mode']:>12}  {rc['regime']:>14}")

## 3. ODE Reference Solver (Ground Truth)

In [ ]:
def ode_rhs_single(t, y, m2, rho_m0, rho_r0, rho_l0):
    """RHS for single-field axion cosmology ODE system.
    State vector y = [a, φ, φ̇]
    """
    a, phi, phi_dot = y
    rho_ax = 0.5 * phi_dot**2 + 0.5 * m2 * phi**2
    rho_m  = rho_m0 / max(a**3, 1e-60)
    rho_r  = rho_r0 / max(a**4, 1e-60)
    H      = np.sqrt(max((rho_ax + rho_m + rho_r + rho_l0) / 3.0, 0.0))
    return [
        a * H,               # da/dt
        phi_dot,             # dφ/dt
        -3.0 * H * phi_dot - m2 * phi,  # dφ̈
    ]


def solve_ode(m_sim: float, phi0: float, a0: float,
              t_end_sim: float,
              rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
              n_points: int = 10000, method: str = 'DOP853') -> dict:
    """
    High-accuracy ODE reference solution in SIMULATION coordinates.
    Always operates at m_sim (not m_target), so it's fast regardless of target mass.
    """
    m2  = m_sim ** 2
    y0  = [a0, phi0, 0.0]   # phi_dot0 = 0 (start from rest)
    t_eval = np.logspace(np.log10(max(a0, 1e-12)), np.log10(t_end_sim), n_points)

    t0 = time.time()
    sol = solve_ivp(
        lambda t, y: ode_rhs_single(t, y, m2, rho_m0, rho_r0, rho_l0),
        [t_eval[0], t_eval[-1]], y0,
        t_eval=t_eval, method=method,
        rtol=1e-10, atol=1e-12, dense_output=False
    )
    dt = time.time() - t0
    print(f"ODE ({method}): {dt:.2f}s | success={sol.success} | nfev={sol.nfev}")
    if not sol.success:
        print(f"  WARNING: {sol.message}")

    a_arr, phi_arr, phidot_arr = sol.y[0], sol.y[1], sol.y[2]
    rho_ax = 0.5 * phidot_arr**2 + 0.5 * m2 * phi_arr**2
    return dict(
        t       = sol.t,
        a       = a_arr,
        phi     = phi_arr,
        phi_dot = phidot_arr,
        rho_ax  = rho_ax,
        rho_m   = rho_m0 / a_arr**3,
        rho_r   = rho_r0 / a_arr**4,
        H       = np.sqrt((rho_ax + rho_m0/a_arr**3 + rho_r0/a_arr**4 + rho_l0) / 3.0),
    )


def plot_ode_reference(ode_sol: dict, mc: dict):
    """Quick 4-panel plot of the ODE reference solution."""
    t = ode_sol['t']
    t_phys = t * mc['lam']   # physical time axis

    fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=100)
    fig.suptitle(f"ODE Reference — m_target={mc['m_target']:.3g}  "
                 f"({mc['regime']}, {mc['mode']})", fontsize=12, fontweight='bold')

    axes[0,0].loglog(t_phys, ode_sol['a'], 'steelblue')
    axes[0,0].set(xlabel='t (physical)', ylabel='a(t)', title='Scale Factor')
    axes[0,0].grid(True, alpha=0.3)

    axes[0,1].semilogx(t_phys, ode_sol['phi'], 'seagreen', lw=0.8)
    axes[0,1].set(xlabel='t (physical)', ylabel='φ(t)',
                  title=f'Axion Field (m_sim={mc["m_sim"]:.2g})')
    axes[0,1].grid(True, alpha=0.3)

    axes[1,0].loglog(ode_sol['a'], ode_sol['rho_ax'] * mc['rho_scale'], 'g-',  label=r'$\rho_{ax}$ (phys)')
    axes[1,0].loglog(ode_sol['a'], ode_sol['rho_m'],  'r-',  label=r'$\rho_m$')
    axes[1,0].loglog(ode_sol['a'], ode_sol['rho_r'],  'b--', label=r'$\rho_r$')
    axes[1,0].axvline(mc['a_osc'], color='grey', ls=':', label=f'a_osc={mc["a_osc"]:.2e}')
    axes[1,0].set(xlabel='a', ylabel='ρ', title='Energy Densities')
    axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

    w_ax = (0.5*ode_sol['phi_dot']**2 - 0.5*mc['m_sim']**2*ode_sol['phi']**2) / (ode_sol['rho_ax'] + 1e-30)
    axes[1,1].semilogx(ode_sol['a'], w_ax, 'purple', lw=0.8)
    axes[1,1].axhline(0,    color='k',    ls='--', lw=0.8, label='w=0 (matter)')
    axes[1,1].axhline(1/3,  color='r',    ls='--', lw=0.8, label='w=1/3 (rad)')
    axes[1,1].axhline(-1,   color='navy', ls='--', lw=0.8, label='w=-1 (Λ)')
    axes[1,1].axvline(mc['a_osc'], color='grey', ls=':', lw=0.8)
    axes[1,1].set(xlabel='a', ylabel='w = p/ρ', title='Equation of State', ylim=(-1.4, 1.4))
    axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 4. PINN Architecture (Single Field)

### การออกแบบ

| ส่วน | รายละเอียด |
|---|---|
| **Hard IC** | $a(0) = a_0$ และ $\phi(0) = \phi_0$ enforce โดยตรงในสถาปัตยกรรม |
| **Fourier Embedding** | trainable scale → auto-adapt ต่อ oscillation frequency ของ $m_{sim}$ |
| **ScaleFactorNet** | $a(t) = a_0 + t \cdot \text{softplus}(\text{net}(t))$ → monotone, positive |
| **AxionFieldNet** | $\phi(t) = \phi_0 + t \cdot \text{net}(t)$ → ResBlocks + sine activation |

In [ ]:
class FourierEmbedding(nn.Module):
    """Random Fourier Features with trainable log-scale."""
    def __init__(self, n_freqs: int = 64, init_scale: float = 10.0, dtype=torch.float64):
        super().__init__()
        B = torch.randn(n_freqs, 1, dtype=dtype)
        self.register_buffer('B', B)
        self.log_scale = nn.Parameter(torch.tensor(float(np.log(init_scale)), dtype=dtype))
        self.out_dim = 2 * n_freqs

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        scale = torch.exp(self.log_scale)
        proj  = 2.0 * np.pi * scale * (t @ self.B.T)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


class SinResBlock(nn.Module):
    """Sine-activation residual block (SIREN-style) for oscillatory solutions."""
    def __init__(self, width: int, dtype=torch.float64):
        super().__init__()
        self.fc1 = nn.Linear(width, width, dtype=dtype)
        self.fc2 = nn.Linear(width, width, dtype=dtype)
        nn.init.uniform_(self.fc1.weight, -np.sqrt(6 / width), np.sqrt(6 / width))
        nn.init.uniform_(self.fc2.weight, -np.sqrt(6 / width), np.sqrt(6 / width))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.fc2(torch.sin(self.fc1(x)))


class ScaleFactorNet(nn.Module):
    """
    Network for a(t).
    Hard IC: a(t) = a0 + t * softplus(net(t))  →  a(0) = a0 exactly.
    Softplus ensures a is monotonically increasing (as expected).
    """
    def __init__(self, a0: float, n_freqs: int = 32, hidden: int = 64, depth: int = 3,
                 init_scale: float = 3.0, dtype=torch.float64):
        super().__init__()
        a0_val = float(a0)
        self.register_buffer('a0', torch.tensor([[a0_val]], dtype=dtype))
        self.fourier = FourierEmbedding(n_freqs, init_scale, dtype=dtype)
        in_dim = self.fourier.out_dim
        layers: list[nn.Module] = [nn.Linear(in_dim, hidden, dtype=dtype), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden, dtype=dtype), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1, dtype=dtype))
        self.net     = nn.Sequential(*layers)
        self.softplus = nn.Softplus()

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.a0 + t * self.softplus(self.net(self.fourier(t)))


class AxionFieldNet(nn.Module):
    """
    Network for φ(t).
    Hard IC: φ(t) = φ0 + t * net(t)  →  φ(0) = φ0 exactly.
    Uses SinResBlocks for oscillatory behaviour.
    Fourier scale is auto-initialised from m_sim.
    """
    def __init__(self, phi0: float, m_sim: float,
                 n_freqs: int = 64, hidden: int = 128, depth: int = 4,
                 dtype=torch.float64):
        super().__init__()
        self.register_buffer('phi0', torch.tensor([[float(phi0)]], dtype=dtype))
        init_scale = float(m_sim) / (2.0 * np.pi)   # tune to oscillation frequency
        self.fourier = FourierEmbedding(n_freqs, init_scale, dtype=dtype)
        in_dim = self.fourier.out_dim
        layers: list[nn.Module] = [nn.Linear(in_dim, hidden, dtype=dtype)]
        for _ in range(depth):
            layers.append(SinResBlock(hidden, dtype=dtype))
        self.net  = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1, dtype=dtype)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        emb   = self.fourier(t)
        delta = self.head(self.net(emb))
        return self.phi0 + t * delta


class SingleAxionPINN(nn.Module):
    """Full model: outputs (a(t), φ(t)) as (N,1) tensors with exact ICs."""
    def __init__(self, m_sim: float, a0: float, phi0: float,
                 n_freqs_a: int = 32, n_freqs_phi: int = 64,
                 hidden_a: int = 64, depth_a: int = 3,
                 hidden_phi: int = 128, depth_phi: int = 4,
                 dtype=torch.float64):
        super().__init__()
        self.a_net   = ScaleFactorNet(a0,   n_freqs_a,   hidden_a,   depth_a,
                                      init_scale=3.0, dtype=dtype)
        self.phi_net = AxionFieldNet(phi0, m_sim, n_freqs_phi, hidden_phi, depth_phi,
                                     dtype=dtype)

    def forward(self, t: torch.Tensor):
        return self.a_net(t), self.phi_net(t)


# ── Quick sanity check ────────────────────────────────────────
_a0_test, _phi0_test, _m_test = 1e-8, 1.0, 70.0
_model_tmp = SingleAxionPINN(_m_test, _a0_test, _phi0_test).to(DEVICE, dtype=DTYPE)
_t0 = torch.zeros(1, 1, dtype=DTYPE, device=DEVICE)
_a0_out, _phi0_out = _model_tmp(_t0)
print(f"IC check:  a(0)  = {_a0_out.item():.4e}  (expected {_a0_test:.4e})")
print(f"IC check:  φ(0)  = {_phi0_out.item():.6f}  (expected {_phi0_test:.6f})")
print(f"Parameters: {sum(p.numel() for p in _model_tmp.parameters() if p.requires_grad):,}")
del _model_tmp

## 5. Physics Loss

ประกอบด้วย 2 residuals (normalized):
- **Friedmann**: $\mathcal{R}_F = \frac{\dot{a} - aH}{aH}$
- **Klein-Gordon**: $\mathcal{R}_{KG} = \frac{\ddot\phi + 3H\dot\phi + m^2\phi}{m^2|\phi| + |\ddot\phi|}$

บวก **causal weighting** ให้ early-time points มี weight มากกว่า

In [ ]:
EPS = 1e-20   # numerical floor

def physics_residuals(model: SingleAxionPINN, t: torch.Tensor,
                      m_sim: float, rho_m0: float, rho_r0: float, rho_l0: float,
                      normalize: bool = True):
    """
    Compute Friedmann and Klein-Gordon residuals at collocation points t.

    normalize=True:
        Friedmann   → dimensionless relative residual ~ ΔH/H
        Klein-Gordon → normalized by characteristic oscillation scale

    Returns: (res_F, res_KG, H)   shapes: (N,1), (N,1), (N,1)
    """
    t = t.requires_grad_(True)
    a, phi = model(t)

    # First and second derivatives via autograd
    a_t    = torch.autograd.grad(a,    t, torch.ones_like(a),    create_graph=True)[0]
    phi_t  = torch.autograd.grad(phi,  t, torch.ones_like(phi),  create_graph=True)[0]
    phi_tt = torch.autograd.grad(phi_t, t, torch.ones_like(phi_t), create_graph=True)[0]

    m2 = float(m_sim) ** 2
    rho_ax  = 0.5 * phi_t**2 + 0.5 * m2 * phi**2
    rho_m_t = rho_m0 * a**(-3)
    rho_r_t = rho_r0 * a**(-4)
    rho_l   = torch.tensor(rho_l0, dtype=t.dtype, device=t.device)
    rho_tot = rho_ax + rho_m_t + rho_r_t + rho_l

    H  = torch.sqrt(torch.clamp(rho_tot / 3.0, min=EPS))
    H3 = torch.sqrt(torch.clamp(3.0 * rho_tot, min=EPS))

    # ── Friedmann residual ──────────────────────────────────────
    res_F = a_t - a * H
    if normalize:
        denom_F = (torch.abs(a * H).detach() + EPS)
        res_F   = res_F / denom_F

    # ── Klein-Gordon residual ───────────────────────────────────
    kg_raw = phi_tt + H3 * phi_t + m2 * phi
    if normalize:
        denom_KG = (m2 * torch.abs(phi) + torch.abs(phi_tt)).detach() + EPS
        res_KG   = kg_raw / denom_KG
    else:
        res_KG = kg_raw

    return res_F, res_KG, H


def causal_weight(t: torch.Tensor, eps: float = 1.0) -> torch.Tensor:
    """Exponential causal weighting: w = exp(-eps * t/t_max)."""
    t_norm = t / (t.max() + EPS)
    return torch.exp(-eps * t_norm).detach()


class AdaptiveWeights:
    """
    Auto-balance Friedmann and KG loss contributions.
    Reweights every `update_every` steps so each loss ≈ their average.
    Uses EMA smoothing to avoid sudden jumps.
    """
    def __init__(self, w_F: float = 1.0, w_KG: float = 1.0,
                 update_every: int = 200, alpha: float = 0.9):
        self.w_F  = w_F
        self.w_KG = w_KG
        self.alpha        = alpha
        self.update_every = update_every
        self._step        = 0

    def step(self, lF: float, lKG: float):
        self._step += 1
        if self._step % self.update_every == 0:
            mean = (lF + lKG) / 2.0 + EPS
            self.w_F  = self.alpha * self.w_F  + (1 - self.alpha) * mean / (lF  + EPS)
            self.w_KG = self.alpha * self.w_KG + (1 - self.alpha) * mean / (lKG + EPS)

    def get(self):
        return self.w_F, self.w_KG


def total_loss(model, t: torch.Tensor, m_sim: float,
               rho_m0: float, rho_r0: float, rho_l0: float,
               w_F: float = 1.0, w_KG: float = 1.0,
               causal_eps: float = 0.5, normalize: bool = True):
    res_F, res_KG, H = physics_residuals(model, t, m_sim, rho_m0, rho_r0, rho_l0, normalize)
    cw = causal_weight(t, causal_eps)
    lF  = w_F  * torch.mean(cw * res_F**2)
    lKG = w_KG * torch.mean(cw * res_KG**2)
    return lF + lKG, lF.item(), lKG.item()

## 6. Training Pipeline (Adam → L-BFGS)

In [ ]:
def sample_collocation(n: int, t_min: float, t_max: float,
                       a_osc: float, t_end_sim: float,
                       dtype, device) -> torch.Tensor:
    """
    Adaptive collocation sampling:
    - 50% log-spaced  (captures early-time dynamics and wide range)
    - 30% clustered near oscillation onset     (where field starts oscillating)
    - 20% uniform                              (fill-in)
    """
    n_log  = int(n * 0.50)
    n_osc  = int(n * 0.30)
    n_uni  = n - n_log - n_osc

    t_safe  = max(t_min, 1e-14)
    t_log   = np.logspace(np.log10(t_safe), np.log10(t_max), n_log)

    # Cluster near the oscillation onset time
    # Rough estimate: t_osc ~ a_osc (in radiation era: a ~ sqrt(2*H0*t))
    t_osc_center = float(a_osc * t_max / max(t_end_sim, 1e-20)) * t_max
    t_osc_center = np.clip(t_osc_center, t_safe, t_max * 0.9)
    t_cluster = np.abs(np.random.normal(t_osc_center, t_osc_center * 0.4, n_osc))
    t_cluster = np.clip(t_cluster, t_safe, t_max)

    t_uni   = np.random.uniform(t_safe, t_max, n_uni)
    t_all   = np.sort(np.concatenate([t_log, t_cluster, t_uni]))
    return torch.tensor(t_all.reshape(-1, 1), dtype=dtype, device=device)


def train(model: SingleAxionPINN, mc: dict,
          rho_m0: float = RHO_M0, rho_r0: float = RHO_R0, rho_l0: float = RHO_L0,
          # Collocation
          n_colloc: int = 4000, batch_size: int = 512,
          # Adam
          adam_epochs: int = 8000, adam_lr: float = 5e-4,
          # L-BFGS
          lbfgs_iters: int = 2000,
          # Hyper
          causal_eps: float = 0.5,
          print_every: int = 1000,
          device=DEVICE, dtype=DTYPE) -> dict:
    """
    Two-stage training:  Adam (exploration) → L-BFGS (refinement).
    Operates entirely in *simulation* coordinates (t_sim ∈ [0, t_end_sim]).
    """
    m_sim     = mc['m_sim']
    t_end_sim = mc['t_end_sim']
    a0        = 1e-8

    model.train()
    history = dict(adam_loss=[], lF=[], lKG=[], w_F=[], w_KG=[], lbfgs_loss=[])
    aw = AdaptiveWeights(update_every=200)

    # ── Phase 1: Adam ─────────────────────────────────────────
    opt_adam = optim.Adam(model.parameters(), lr=adam_lr, eps=1e-9)
    sched    = CosineAnnealingLR(opt_adam, T_max=adam_epochs, eta_min=adam_lr * 1e-3)

    print("=" * 62)
    print(f"  Phase 1 – Adam  |  m_sim={m_sim:.3g}  t_end_sim={t_end_sim:.3e}"
          f"  [{mc['mode']}]")
    print("=" * 62)
    t0_train = time.time()

    for epoch in range(adam_epochs):
        t_all = sample_collocation(n_colloc, 0.0, t_end_sim,
                                   mc['a_osc'], t_end_sim, dtype, device)
        idx = torch.randperm(n_colloc, device=device)[:batch_size]
        t_b = t_all[idx].clone()

        w_F, w_KG = aw.get()
        opt_adam.zero_grad()
        loss, lF, lKG = total_loss(model, t_b, m_sim,
                                    rho_m0, rho_r0, rho_l0,
                                    w_F=w_F, w_KG=w_KG,
                                    causal_eps=causal_eps, normalize=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt_adam.step()
        sched.step()
        aw.step(lF, lKG)

        history['adam_loss'].append(loss.item())
        history['lF'].append(lF)
        history['lKG'].append(lKG)
        history['w_F'].append(w_F)
        history['w_KG'].append(w_KG)

        if epoch % print_every == 0 or epoch == adam_epochs - 1:
            lr_now = opt_adam.param_groups[0]['lr']
            print(f"  [{epoch:5d}/{adam_epochs}]  loss={loss.item():.3e}  "
                  f"lF={lF:.3e}  lKG={lKG:.3e}  "
                  f"wF={w_F:.2f}  wKG={w_KG:.2f}  lr={lr_now:.2e}")

    print(f"  Adam done in {time.time()-t0_train:.1f}s")

    # ── Phase 2: L-BFGS ──────────────────────────────────────
    print("\n" + "=" * 62)
    print("  Phase 2 – L-BFGS Refinement")
    print("=" * 62)

    t_full = sample_collocation(n_colloc, 0.0, t_end_sim,
                                mc['a_osc'], t_end_sim, dtype, device)
    t_full.requires_grad_(True)
    w_F, w_KG = aw.get()
    lbfgs_buf: list[float] = []

    opt_lbfgs = optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=lbfgs_iters,
        tolerance_grad=1e-11, tolerance_change=1e-12,
        history_size=150, line_search_fn='strong_wolfe'
    )

    def _closure():
        opt_lbfgs.zero_grad()
        loss, lF, lKG = total_loss(model, t_full, m_sim,
                                    rho_m0, rho_r0, rho_l0,
                                    w_F=w_F, w_KG=w_KG,
                                    causal_eps=causal_eps, normalize=True)
        loss.backward()
        lbfgs_buf.append(loss.item())
        return loss

    t0_lb = time.time()
    opt_lbfgs.step(_closure)
    history['lbfgs_loss'] = lbfgs_buf
    print(f"  L-BFGS done in {time.time()-t0_lb:.1f}s  |  "
          f"final loss = {lbfgs_buf[-1]:.3e}")
    return history

## 7. Evaluation & Metrics

ฟังก์ชัน evaluate คืนค่า prediction ทั้งในเชิง simulation และ physical coordinates  
พร้อมตรวจสอบ Friedmann constraint violation ตลอด trajectory

In [ ]:
def evaluate(model: SingleAxionPINN, ode_sol: dict, mc: dict,
             n_eval: int = 5000,
             rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
             device=DEVICE, dtype=DTYPE) -> dict:
    """
    Evaluate PINN on a dense grid and compare against ODE reference.

    Returns a dict with:
        t_sim, t_phys : time arrays (simulation and physical)
        a_pred, phi_pred : PINN predictions
        a_ode,  phi_ode  : ODE interpolated to same grid
        rho_ax_pred      : axion energy density from PINN
        metrics          : L2-relative, max-abs, Friedmann violation
    """
    model.eval()
    m_sim     = mc['m_sim']
    t_end_sim = mc['t_end_sim']
    lam       = mc['lam']
    a0        = 1e-8

    t_sim  = np.logspace(np.log10(max(a0, 1e-13)), np.log10(t_end_sim), n_eval)
    t_phys = t_sim * lam

    # ── PINN predictions ─────────────────────────────────────
    t_ten = torch.tensor(t_sim.reshape(-1, 1), dtype=dtype, device=device)
    with torch.no_grad():
        a_pred   = model.a_net(t_ten).cpu().numpy().flatten()
        phi_pred = model.phi_net(t_ten).cpu().numpy().flatten()

    # ── ODE interpolation ────────────────────────────────────
    t_ref = ode_sol['t']
    a_ode   = interp1d(t_ref, ode_sol['a'],   kind='cubic', fill_value='extrapolate')(t_sim)
    phi_ode = interp1d(t_ref, ode_sol['phi'], kind='cubic', fill_value='extrapolate')(t_sim)

    # ── Axion energy density from PINN ───────────────────────
    t_g = t_ten.clone().requires_grad_(True)
    with torch.enable_grad():
        phi_g = model.phi_net(t_g)
        dphi  = torch.autograd.grad(phi_g.sum(), t_g, create_graph=False)[0]
    phi_np  = phi_g.detach().cpu().numpy().flatten()
    dphi_np = dphi.detach().cpu().numpy().flatten()
    rho_ax_sim  = 0.5 * dphi_np**2 + 0.5 * m_sim**2 * phi_np**2
    rho_ax_phys = rho_ax_sim * mc['rho_scale']

    # ── Friedmann violation ──────────────────────────────────
    t_fv = t_ten.requires_grad_(True)
    a_fv, phi_fv = model(t_fv)
    a_t   = torch.autograd.grad(a_fv,   t_fv, torch.ones_like(a_fv),   create_graph=False)[0]
    phi_t = torch.autograd.grad(phi_fv, t_fv, torch.ones_like(phi_fv), create_graph=False)[0]
    m2 = m_sim**2
    rho_ax_fv = 0.5 * phi_t**2 + 0.5 * m2 * phi_fv**2
    rho_tot_fv = (rho_ax_fv + rho_m0 * a_fv**(-3) + rho_r0 * a_fv**(-4)
                  + torch.tensor(rho_l0, dtype=dtype, device=device))
    H_fv  = torch.sqrt(torch.clamp(rho_tot_fv / 3.0, min=EPS))
    aH_fv = a_fv * H_fv
    viol  = (torch.abs(a_t - aH_fv) / (torch.abs(aH_fv) + EPS)).detach().cpu().numpy().flatten()

    # ── Error metrics ────────────────────────────────────────
    def l2rel(pred, ref):
        return np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + EPS)
    def maxabs(pred, ref):
        return np.max(np.abs(pred - ref))

    metrics = {
        'L2_rel_a'   : l2rel(a_pred,   a_ode),
        'MaxAbs_a'   : maxabs(a_pred,  a_ode),
        'L2_rel_phi' : l2rel(phi_pred, phi_ode),
        'MaxAbs_phi' : maxabs(phi_pred, phi_ode),
        'viol_mean'  : float(np.mean(viol)),
        'viol_max'   : float(np.max(viol)),
    }

    print("\n" + "─" * 48)
    print("  Evaluation Metrics  (PINN vs ODE)")
    print("─" * 48)
    for k, v in metrics.items():
        print(f"  {k:<20} = {v:.4e}")
    print("─" * 48)

    return dict(
        t_sim=t_sim, t_phys=t_phys,
        a_pred=a_pred,   phi_pred=phi_pred,
        a_ode=a_ode,     phi_ode=phi_ode,
        rho_ax_phys=rho_ax_phys,
        friedmann_viol=viol,
        metrics=metrics,
    )

## 8. Visualization

In [ ]:
def plot_results(ev: dict, history: dict, mc: dict,
                 save_dir: str = 'results_single_field'):
    """
    Comprehensive 6-panel figure:
      Row 1: a(t) comparison | a(t) relative error | φ(t) comparison
      Row 2: Training loss   | Friedmann violation  | Error metrics table
    """
    os.makedirs(save_dir, exist_ok=True)

    t_phys = ev['t_phys']
    a_pred, a_ode   = ev['a_pred'],   ev['a_ode']
    phi_pred, phi_ode = ev['phi_pred'], ev['phi_ode']
    met = ev['metrics']

    fig = plt.figure(figsize=(16, 11), dpi=110)
    gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

    # ── a(t) ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ax.loglog(t_phys, a_ode,  'k--', lw=1.8, label='ODE (ref)')
    ax.loglog(t_phys, a_pred, 'C1-', lw=1.5, label='PINN')
    ax.set(xlabel='t (physical)', ylabel='a(t)',
           title=f'Scale Factor\n$L_2^{{rel}}$ = {met["L2_rel_a"]:.2e}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # ── a(t) relative error ──────────────────────────────────
    ax = fig.add_subplot(gs[0, 1])
    err_a = np.abs(a_pred - a_ode) / (np.abs(a_ode) + EPS) * 100
    ax.semilogx(t_phys, err_a, 'C3', lw=1)
    ax.axhline(1.0, color='grey', ls='--', lw=1, label='1%')
    ax.set(xlabel='t (physical)', ylabel='Relative Error (%)',
           title='a(t) Error')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # ── φ(t) ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 2])
    # Show only early portion if many oscillations (keep plot readable)
    t_show = t_phys
    mask   = np.ones(len(t_phys), bool)
    if mc['n_osc'] > 20:
        # Show up to ~5 oscillation periods
        t_cut  = 5 * mc['lam'] * (2 * np.pi / mc['m_sim'])
        mask   = t_phys <= max(t_cut, t_phys[len(t_phys)//5])
    ax.plot(t_show[mask], phi_ode[mask],  'k--', lw=1.5, label='ODE')
    ax.plot(t_show[mask], phi_pred[mask], 'C0-', lw=1.2, label='PINN')
    ax.set_xscale('log')
    ax.set(xlabel='t (physical)', ylabel='φ(t)',
           title=f'Axion Field\n$L_2^{{rel}}$ = {met["L2_rel_phi"]:.2e}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # ── Training loss ────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 0])
    epochs = np.arange(1, len(history['adam_loss']) + 1)
    ax.semilogy(epochs, history['adam_loss'], 'steelblue', lw=1.2, label='Total')
    ax.semilogy(epochs, history['lF'],        'orange',    lw=1.0, ls='--', label='Friedmann')
    ax.semilogy(epochs, history['lKG'],       'green',     lw=1.0, ls=':',  label='KG')
    ax.set(xlabel='Adam epoch', ylabel='Loss', title='Training Loss (Adam)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Friedmann violation ──────────────────────────────────
    ax = fig.add_subplot(gs[1, 1])
    ax.semilogx(t_phys, ev['friedmann_viol'] * 100, 'coral', lw=1)
    ax.axhline(1.0, color='grey', ls='--', lw=1, label='1% threshold')
    ax.set(xlabel='t (physical)', ylabel='|ΔH/H| (%)',
           title=f'Friedmann Violation\nmean={met["viol_mean"]*100:.3f}%  '
                 f'max={met["viol_max"]*100:.3f}%')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Metrics table ────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 2])
    info_lines = [
        f"m_target = {mc['m_target']:.3g}  ({mc['regime']})",
        f"mode     = {mc['mode']}",
        f"m_sim    = {mc['m_sim']:.3g}  λ={mc['lam']:.3g}" if mc['mode']=='rescaling' else '',
        f"m_a      ≈ {mc['m_target'] * H0_eV:.3e} eV",
        "",
    ] + [f"{k}: {v:.3e}" for k, v in met.items()]
    ax.text(0.05, 0.95, '\n'.join(info_lines),
            ha='left', va='top', transform=ax.transAxes,
            fontfamily='monospace', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#f7f7e8', alpha=0.8))
    ax.set_axis_off(); ax.set_title('Summary')

    fig.suptitle(
        f'Single-Field Axion PINN  |  m={mc["m_target"]:.3g}  '
        f'({mc["m_target"]*H0_eV:.2e} eV)',
        fontsize=13, fontweight='bold'
    )
    tag  = f"m{mc['m_target']:.0e}".replace('+', '').replace('e0', 'e')
    path = os.path.join(save_dir, f'{tag}_results.png')
    plt.savefig(path, bbox_inches='tight', dpi=120)
    plt.show()
    print(f"Saved → {path}")


def plot_training_detail(history: dict, mc: dict):
    """Extra plot: loss + adaptive weights + L-BFGS curve."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=100)

    epochs = np.arange(1, len(history['adam_loss']) + 1)

    axes[0].semilogy(epochs, history['adam_loss'], lw=1.2)
    axes[0].set(xlabel='epoch', ylabel='loss', title='Adam: Total Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['w_F'],  label='$w_F$  (Friedmann)')
    axes[1].plot(epochs, history['w_KG'], label='$w_{KG}$ (Klein-Gordon)')
    axes[1].set(xlabel='epoch', ylabel='weight', title='Self-Adaptive Weights')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

    if history['lbfgs_loss']:
        axes[2].semilogy(history['lbfgs_loss'], color='darkorange', lw=1.2)
        axes[2].set(xlabel='L-BFGS iteration', ylabel='loss', title='L-BFGS Refinement')
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No L-BFGS data', ha='center', va='center',
                     transform=axes[2].transAxes)
        axes[2].set_axis_off()

    plt.suptitle(f'Training Detail  |  m_sim={mc["m_sim"]:.3g}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 9. Master Entry Point: `run_experiment`

ฟังก์ชัน **เดียว** ที่รผู้ใช้เรียก — ระบุแค่ `m_target` แล้วทุกอย่างทำงานอัตโนมัติ:

```python
results = run_experiment(m_target=70.0)        # direct mode
results = run_experiment(m_target=1e6)         # auto-rescaling mode
results = run_experiment(m_target=1.0, phi0=2.0)  # slow-roll regime
```

In [ ]:
def run_experiment(
    # ── Physics ──────────────────────────────────────────────────
    m_target: float = 70.0,   # axion mass in code units (H0 = 1)
    phi0: float     = 1.0,    # initial field value
    a0: float       = 1e-8,   # initial scale factor
    t_end: float    = 1.0,    # end of integration [code time, H0^-1]
    # ── Network size (auto-scaled by regime) ─────────────────────
    n_freqs_a:   int = None,
    n_freqs_phi: int = None,
    hidden_a:    int = None,
    hidden_phi:  int = None,
    depth_a:     int = 3,
    depth_phi:   int = 4,
    # ── Training ─────────────────────────────────────────────────
    adam_epochs: int   = 8000,
    adam_lr:     float = 5e-4,
    lbfgs_iters: int   = 2000,
    n_colloc:    int   = 4000,
    batch_size:  int   = 512,
    causal_eps:  float = 0.5,
    print_every: int   = 1000,
    # ── Misc ─────────────────────────────────────────────────────
    save_dir: str = 'results_single_field',
    device = DEVICE,
    dtype  = DTYPE,
) -> dict:
    """
    Full single-axion PINN pipeline.

    Automatically:
    1. Analyses mass regime
    2. Chooses direct or rescaling simulation strategy
    3. Scales network architecture to match difficulty
    4. Solves ODE reference
    5. Trains PINN
    6. Evaluates and plots results

    Returns dict with keys: mc, model, ode_sol, history, eval
    """
    # ── Step 1: Mass regime analysis ─────────────────────────
    mc = analyze_mass_regime(m_target, t_end)
    print_mass_summary(mc)

    # ── Step 2: Auto-scale network if not specified ───────────
    # Harder regimes (more oscillations) → bigger network
    n_osc = mc['n_osc']
    if n_freqs_a   is None: n_freqs_a   = 32  if n_osc < 50  else 48
    if n_freqs_phi is None: n_freqs_phi = 64  if n_osc < 50  else (96  if n_osc < 300 else 128)
    if hidden_a    is None: hidden_a    = 64  if n_osc < 50  else 96
    if hidden_phi  is None: hidden_phi  = 128 if n_osc < 50  else (192 if n_osc < 300 else 256)

    print(f"\nNetwork:  freqs=({n_freqs_a},{n_freqs_phi})  "
          f"hidden=({hidden_a},{hidden_phi})  depth=({depth_a},{depth_phi})")

    # ── Step 3: ODE reference in simulation coordinates ──────
    print(f"\nSolving ODE reference (m_sim={mc['m_sim']:.3g}) ...")
    ode_sol = solve_ode(mc['m_sim'], phi0, a0, mc['t_end_sim'])
    plot_ode_reference(ode_sol, mc)

    # ── Step 4: Build model ───────────────────────────────────
    model = SingleAxionPINN(
        m_sim     = mc['m_sim'],
        a0        = a0,
        phi0      = phi0,
        n_freqs_a   = n_freqs_a,
        n_freqs_phi = n_freqs_phi,
        hidden_a    = hidden_a,
        depth_a     = depth_a,
        hidden_phi  = hidden_phi,
        depth_phi   = depth_phi,
        dtype = dtype,
    ).to(device, dtype=dtype)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {n_params:,}")

    # ── Step 5: Train ─────────────────────────────────────────
    history = train(
        model, mc,
        adam_epochs = adam_epochs,
        adam_lr     = adam_lr,
        lbfgs_iters = lbfgs_iters,
        n_colloc    = n_colloc,
        batch_size  = batch_size,
        causal_eps  = causal_eps,
        print_every = print_every,
        device = device, dtype = dtype,
    )

    # ── Step 6: Evaluate & visualise ─────────────────────────
    ev = evaluate(model, ode_sol, mc, device=device, dtype=dtype)
    plot_results(ev, history, mc, save_dir=save_dir)
    plot_training_detail(history, mc)

    return dict(mc=mc, model=model, ode_sol=ode_sol, history=history, eval=ev)

## 10. Experiment A — Medium Mass: m = 70 H₀ (direct mode)

มวลปานกลาง (~10⁻³¹ eV) — axion เริ่มสั่นใน radiation era  
PINN train ตรงๆ ไม่ต้อง rescale

In [ ]:
results_m70 = run_experiment(
    m_target    = 70.0,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    adam_epochs = 8000,
    adam_lr     = 5e-4,
    lbfgs_iters = 2000,
    n_colloc    = 4000,
    causal_eps  = 0.5,
    print_every = 1000,
)

## 11. Experiment B — Slow-Roll Regime: m = 3 H₀

Fuzzy DM ที่มวลต่ำมาก — axion practically frozen ตลอด  
ทดสอบว่า PINN เรียนรู้ slow-roll behavior ได้ถูกต้องหรือไม่

In [ ]:
results_m3 = run_experiment(
    m_target    = 3.0,
    phi0        = 1.5,
    a0          = 1e-8,
    t_end       = 1.0,
    adam_epochs = 6000,
    adam_lr     = 5e-4,
    lbfgs_iters = 1500,
    n_colloc    = 3000,
    causal_eps  = 0.3,
    print_every = 1000,
)

## 12. Experiment C — High Mass: m = 1×10⁶ H₀ (rescaling mode)

ULA ระดับ ~10⁻²⁷ eV — oscillation เร็วมาก  
ระบบ auto-rescale และ simulate ที่ m_sim ที่เหมาะสม

In [ ]:
results_m1e6 = run_experiment(
    m_target    = 1e6,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    adam_epochs = 12000,
    adam_lr     = 3e-4,
    lbfgs_iters = 3000,
    n_colloc    = 5000,
    causal_eps  = 1.0,
    print_every = 2000,
)

## 13. Mass Sweep — Error vs Mass

สแกนมวลหลายค่าและแสดง L₂ relative error ของ a(t) และ φ(t) เป็นฟังก์ชันของมวล  
ใช้ training แบบสั้น (quick_run) เพื่อความเร็ว

In [ ]:
MASS_SWEEP = [1.0, 5.0, 20.0, 70.0, 200.0, 500.0, 2000.0, 1e5, 1e7]

sweep_results = []

for m in MASS_SWEEP:
    print(f"\n{'='*55}")
    print(f"  SWEEP:  m_target = {m:.2e}")
    print(f"{'='*55}")
    try:
        mc = analyze_mass_regime(m, t_end=1.0)

        # Quick ODE reference
        ode = solve_ode(mc['m_sim'], 1.0, 1e-8, mc['t_end_sim'], n_points=5000)

        # Small model for sweep
        mdl = SingleAxionPINN(
            m_sim=mc['m_sim'], a0=1e-8, phi0=1.0,
            n_freqs_a=32, n_freqs_phi=64, hidden_a=64, hidden_phi=128,
            depth_a=3, depth_phi=4, dtype=DTYPE,
        ).to(DEVICE, dtype=DTYPE)

        # Quick training (fewer epochs for sweep)
        hist = train(mdl, mc,
                     adam_epochs=5000, adam_lr=5e-4, lbfgs_iters=1000,
                     n_colloc=3000, batch_size=512, causal_eps=0.5,
                     print_every=5000,   # suppress per-epoch output
                     device=DEVICE, dtype=DTYPE)

        ev = evaluate(mdl, ode, mc, n_eval=3000, device=DEVICE, dtype=DTYPE)
        sweep_results.append({
            'm': m,
            'regime': mc['regime'],
            'mode': mc['mode'],
            'L2_a': ev['metrics']['L2_rel_a'],
            'L2_phi': ev['metrics']['L2_rel_phi'],
            'viol': ev['metrics']['viol_mean'],
        })
        del mdl
        torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    except Exception as e:
        print(f"  ERROR at m={m:.2e}: {e}")
        sweep_results.append({'m': m, 'regime': '?', 'mode': '?',
                               'L2_a': np.nan, 'L2_phi': np.nan, 'viol': np.nan})

# ── Plot sweep results ────────────────────────────────────────
masses  = [r['m']      for r in sweep_results]
L2_a    = [r['L2_a']   for r in sweep_results]
L2_phi  = [r['L2_phi'] for r in sweep_results]
viols   = [r['viol']   for r in sweep_results]
regimes = [r['regime'] for r in sweep_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=110)

# Color by regime
regime_colors = {'slow-roll': 'C2', 'onset': 'C0',
                 'oscillating': 'C1', 'rapid-osc': 'C3', '?': 'grey'}
colors = [regime_colors.get(r, 'grey') for r in regimes]

for ax, vals, ylabel, title in zip(
    axes,
    [L2_a, L2_phi],
    ['$L_2$ rel. error', '$L_2$ rel. error'],
    ['Scale Factor a(t)', 'Axion Field φ(t)'],
):
    valid = [(m, v) for m, v in zip(masses, vals) if not np.isnan(v)]
    ax.scatter([v[0] for v in valid], [v[1] for v in valid],
               c=[colors[masses.index(v[0])] for v in valid], s=80, zorder=5)
    ax.plot([v[0] for v in valid], [v[1] for v in valid], 'k-', lw=0.7, alpha=0.4)
    ax.axhline(0.01, color='grey', ls='--', lw=1, label='1% target')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set(xlabel='m_target (code units)', ylabel=ylabel, title=f'L₂ Error: {title}')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

# Legend for regimes
from matplotlib.lines import Line2D
legend_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                          markersize=10, label=r)
                  for r, c in regime_colors.items() if r != '?']
axes[0].legend(handles=legend_handles, fontsize=8, title='Regime')

plt.suptitle('Mass Sweep: PINN Accuracy across Regimes', fontsize=12, fontweight='bold')
plt.tight_layout()
os.makedirs('results_single_field', exist_ok=True)
plt.savefig('results_single_field/mass_sweep.png', bbox_inches='tight', dpi=120)
plt.show()

# Table
print(f"\n{'m_target':>12}  {'regime':>14}  {'mode':>12}  {'L2_a':>10}  {'L2_phi':>10}  {'viol':>10}")
print("─" * 70)
for r in sweep_results:
    print(f"{r['m']:>12.2e}  {r['regime']:>14}  {r['mode']:>12}  "
          f"{r['L2_a']:>10.3e}  {r['L2_phi']:>10.3e}  {r['viol']:>10.3e}")

## 14. Save & Load

บันทึก model พร้อม mass config และ metrics ไว้ใน checkpoint  
โหลดกลับมา evaluate ต่อได้โดยไม่ต้อง retrain

In [ ]:
def save_checkpoint(results: dict, path: str):
    """Save model + config + metrics to a .pt checkpoint."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    torch.save({
        'model_state': results['model'].state_dict(),
        'mc':          results['mc'],
        'metrics':     results['eval']['metrics'],
    }, path)
    print(f"Saved → {path}")


def load_checkpoint(path: str, device=DEVICE, dtype=DTYPE) -> tuple:
    """
    Load checkpoint. Returns (model, mc, metrics).
    Note: phi0 and a0 are stored in the model's buffers, so no need to pass them.
    """
    ckpt  = torch.load(path, map_location=device)
    mc    = ckpt['mc']
    # Reconstruct model with same architecture defaults
    mdl = SingleAxionPINN(
        m_sim  = mc['m_sim'],
        a0     = 1e-8,   # stored as buffer; will be overwritten by state_dict
        phi0   = 1.0,    # same
        dtype  = dtype,
    ).to(device, dtype=dtype)
    mdl.load_state_dict(ckpt['model_state'])
    mdl.eval()
    print(f"Loaded ← {path}")
    print(f"  m_target={mc['m_target']:.3g}  regime={mc['regime']}  mode={mc['mode']}")
    print(f"  Saved metrics: {ckpt['metrics']}")
    return mdl, mc, ckpt['metrics']


# ── Save Experiment A ─────────────────────────────────────────
save_checkpoint(results_m70,  'results_single_field/checkpoints/model_m70.pt')

# ── Example: reload and quick-evaluate ───────────────────────
mdl_loaded, mc_loaded, met_loaded = load_checkpoint(
    'results_single_field/checkpoints/model_m70.pt'
)
print("\nLoaded model IC check:")
t0_b = torch.zeros(1, 1, dtype=DTYPE, device=DEVICE)
a0_out, phi0_out = mdl_loaded(t0_b)
print(f"  a(0)  = {a0_out.item():.4e}")
print(f"  φ(0)  = {phi0_out.item():.6f}")

## 15. Summary & ขั้นตอนต่อไป

---

### สิ่งที่มีในไฟล์นี้

| Component | รายละเอียด | สถานะ |
|---|---|---|
| `analyze_mass_regime()` | auto-classify + choose direct/rescaling | ✅ |
| `solve_ode()` | DOP853 ground truth ใน sim coords | ✅ |
| `SingleAxionPINN` | Hard IC, Fourier+ResBlock, float64 | ✅ |
| `physics_residuals()` | normalized Friedmann + KG | ✅ |
| `AdaptiveWeights` | auto-rebalance loss terms | ✅ |
| `train()` | Adam + L-BFGS, causal weighting | ✅ |
| `evaluate()` | metrics + Friedmann violation check | ✅ |
| `run_experiment()` | single-call master pipeline | ✅ |
| Mass sweep | error vs mass across all regimes | ✅ |
| Save/Load | checkpoint with config + metrics | ✅ |

---

### การปรับพารามิเตอร์

```python
# เปลี่ยนมวลได้เลย — ระบบจัดการ rescaling ให้อัตโนมัติ
results = run_experiment(m_target=1e9, phi0=0.5, adam_epochs=15000)

# ต้องการ network ใหญ่ขึ้น (มวลสูง + accuracy สูง)
results = run_experiment(
    m_target=1e8,
    n_freqs_phi=128, hidden_phi=256, depth_phi=6,
    adam_epochs=20000, lbfgs_iters=5000,
)
```

### แนวทางพัฒนาต่อ

1. **Potential beyond $m^2\phi^2$**: เพิ่ม anharmonic term $\lambda\phi^4$ ใน `physics_residuals`
2. **Time-domain decomposition**: แบ่ง $t \in [0,1]$ เป็น sub-intervals แล้วต่อกัน (XPINNs)  
3. **Uncertainty quantification**: ใส่ Bayesian layer หรือ ensemble
4. **Kähler coupling**: ขยายกลับไป multi-field พร้อม off-diagonal metric